<a href="https://colab.research.google.com/github/Abdullah200401/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah200401/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_Token")
login(token=HF_TOKEN)

content_df = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)["train"].to_pandas()

print("Dataset loaded successfully!")
print("Rows:", len(content_df))
print("Columns:", len(content_df.columns))

Dataset loaded successfully!
Rows: 519606
Columns: 26


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 1) Ranked actions + reason codes

The ranked queue is intended to help a human reviewer prioritize content refresh opportunities. Each item receives a priority score and a reason code explaining why it was selected.

The main reason codes are:

* **STALE_HIGH_VOLUME** — content appears stale and has relatively high search demand.
* **STALE** — content appears stale and may need refresh review.
* **HIGH_VOLUME** — content has relatively high search demand but does not meet the staleness threshold.
* **LOW_SIGNAL** — no strong refresh signal was identified.

The queue is a prioritization tool rather than an automatic decision system. Higher-ranked items should be reviewed first, but the ranking does not by itself mean that a page should be changed.


In [6]:
import pandas as pd
import numpy as np

# Work from the content dataset
playbook_df = content_df.copy()

# Decision date
decision_date = pd.Timestamp("2026-06-30")

# Calculate staleness
last_optimized = pd.to_datetime(
    playbook_df["last_optimized_date"],
    errors="coerce"
)

playbook_df["days_since_optimized"] = (
    decision_date - last_optimized
).dt.days

# Missing optimization date = very stale
playbook_df["days_since_optimized"] = (
    playbook_df["days_since_optimized"]
    .fillna(9999)
)

# Median search volume
volume_median = playbook_df["search_volume"].median()

# Signals
playbook_df["stale_flag"] = (
    playbook_df["days_since_optimized"] >= 180
)

playbook_df["high_volume_flag"] = (
    playbook_df["search_volume"] >= volume_median
)

# Score
playbook_df["priority_score"] = (
    playbook_df["stale_flag"].astype(int) * 2
    + playbook_df["high_volume_flag"].astype(int)
)

# Reason codes
def get_reason(row):
    if row["stale_flag"] and row["high_volume_flag"]:
        return "STALE_HIGH_VOLUME"
    elif row["stale_flag"]:
        return "STALE"
    elif row["high_volume_flag"]:
        return "HIGH_VOLUME"
    else:
        return "LOW_SIGNAL"

playbook_df["reason_code"] = playbook_df.apply(
    get_reason,
    axis=1
)

# Action
action_map = {
    "STALE_HIGH_VOLUME": "REFRESH",
    "STALE": "REVIEW_REFRESH",
    "HIGH_VOLUME": "MONITOR",
    "LOW_SIGNAL": "NO_ACTION"
}

playbook_df["recommended_action"] = (
    playbook_df["reason_code"].map(action_map)
)

# Rank highest priority first
playbook_df = playbook_df.sort_values(
    ["priority_score", "search_volume"],
    ascending=[False, False]
).reset_index(drop=True)

playbook_df["rank"] = np.arange(1, len(playbook_df) + 1)

# Display top 20
display(
    playbook_df[
        [
            "rank",
            "priority_score",
            "reason_code",
            "recommended_action",
            "search_volume",
            "days_since_optimized"
        ]
    ].head(20)
)

,rank,priority_score,reason_code,recommended_action,search_volume,days_since_optimized
0,1,3,STALE_HIGH_VOLUME,REFRESH,368000.0,9999.0
1,2,3,STALE_HIGH_VOLUME,REFRESH,368000.0,9999.0
2,3,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
3,4,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
4,5,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
5,6,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
6,7,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
7,8,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
8,9,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
9,10,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### 2) Intended use and limits

The action playbook is intended to support human reviewers in prioritizing content that may deserve refresh or further investigation.

The ranked score should be used as a decision-support signal, not as an automatic instruction to change content. A high score indicates that the available signals suggest higher priority for review, but it does not prove that refreshing the page will improve performance.

The playbook is based on historical and observational data. It may be affected by differences between clients, content types, search demand, and other factors that are not fully captured by the model.

Therefore, the output should be used for prioritization and investigation rather than guaranteed prediction of future traffic, rankings, or business outcomes.


In [8]:
# Section 2: Basic playbook checks

print("Total content items:", len(playbook_df))
print("Reason codes:")
print(playbook_df["reason_code"].value_counts())

print("\nRecommended actions:")
print(playbook_df["recommended_action"].value_counts())

print("\nPriority score range:")
print(
    playbook_df["priority_score"].min(),
    "to",
    playbook_df["priority_score"].max()
)

Total content items: 519606
Reason codes:
reason_code
STALE                289930
STALE_HIGH_VOLUME    184280
HIGH_VOLUME           29073
LOW_SIGNAL            16323
Name: count, dtype: int64

Recommended actions:
recommended_action
REVIEW_REFRESH    289930
REFRESH           184280
MONITOR            29073
NO_ACTION          16323
Name: count, dtype: int64

Priority score range:
0 to 3


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3) Human review + the no-go list

Every ranked recommendation should be reviewed by a human before any content change is made.

The reviewer should check the page's search intent, current rankings, recent performance, content quality, freshness, business relevance, and whether the recommended action makes sense in context.

The following should **not** be automated:

* Publishing or deleting content.
* Changing search intent or page purpose.
* Rewriting important claims without fact-checking.
* Making medical, legal, financial, or other high-stakes claims.
* Redirecting or canonicalizing pages automatically.
* Making changes solely because a page has a high priority score.
* Treating the model score as proof that a refresh will improve performance.

The model identifies candidates for review; a human makes the final decision.


In [10]:
# Section 3: Human-review queue

review_queue = playbook_df[
    playbook_df["recommended_action"] != "NO_ACTION"
].copy()

print("Items requiring human review:", len(review_queue))

display(
    review_queue[
        [
            "rank",
            "reason_code",
            "recommended_action",
            "search_volume",
            "days_since_optimized"
        ]
    ].head(20)
)

Items requiring human review: 503283


,rank,reason_code,recommended_action,search_volume,days_since_optimized
0,1,STALE_HIGH_VOLUME,REFRESH,368000.0,9999.0
1,2,STALE_HIGH_VOLUME,REFRESH,368000.0,9999.0
2,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
3,4,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
4,5,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
5,6,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
6,7,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
7,8,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
8,9,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
9,10,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 4) Monitoring / retrain triggers

The playbook should be monitored periodically rather than treated as a permanent ranking system.

I would review the output and underlying data when search demand, content mix, or client coverage changes substantially.

Potential retrain or review triggers include:

* A meaningful change in the distribution of important input features.
* A substantial increase in missing or invalid data.
* Model performance declining on newly observed outcomes.
* A change in the relationship between priority scores and observed content performance.
* Major changes in search behavior or the underlying data pipeline.

These triggers are intended as practical monitoring signals rather than strict production thresholds. Any retraining should first be validated on a fresh and appropriately separated evaluation set.


In [12]:
# Section 4: Basic monitoring checks

print("Monitoring summary")
print("------------------")

print("Total items:", len(playbook_df))
print("Missing search volume:",
      playbook_df["search_volume"].isna().sum())

print("Missing optimization date:",
      playbook_df["last_optimized_date"].isna().sum())

print("\nPriority score distribution:")
print(playbook_df["priority_score"].value_counts().sort_index())

Monitoring summary
------------------
Total items: 519606
Missing search volume: 142622
Missing optimization date: 474210

Priority score distribution:
priority_score
0     16323
1     29073
2    289930
3    184280
Name: count, dtype: int64


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### 5) Exports for the paper

The notebook exports the ranked action queue so that the recommendations and analysis in the research paper can be traced back to the notebook output.

The exported queue contains the ranking, priority score, reason code, recommended action, and supporting signals used for prioritization.

The CSV is generated by the notebook and stored under `work/outputs/`. It is not treated as a production dataset or automatically deployed system.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 5: Export ranked queue

import os

os.makedirs("/content/work/outputs", exist_ok=True)

export_columns = [
    "rank",
    "priority_score",
    "reason_code",
    "recommended_action",
    "search_volume",
    "days_since_optimized"
]

queue_export = playbook_df[export_columns].copy()

output_path = "/content/work/outputs/action_playbook_queue.csv"

queue_export.to_csv(
    output_path,
    index=False
)

print("Export created:")
print(output_path)

print("\nRows exported:", len(queue_export))

display(queue_export.head(10))

Export created:
/content/work/outputs/action_playbook_queue.csv

Rows exported: 519606


,rank,priority_score,reason_code,recommended_action,search_volume,days_since_optimized
0,1,3,STALE_HIGH_VOLUME,REFRESH,368000.0,9999.0
1,2,3,STALE_HIGH_VOLUME,REFRESH,368000.0,9999.0
2,3,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
3,4,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
4,5,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
5,6,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
6,7,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
7,8,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
8,9,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0
9,10,3,STALE_HIGH_VOLUME,REFRESH,301000.0,9999.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.